# MusicPulse — Firestore et Redis appliqués aux données musicales

Notebook de démarche destiné au livrable. L'objectif principal est d'expliquer le stockage documentaire durable dans Firestore et les structures rapides de Redis. La recommandation musicale sert de cas d'usage.

Le notebook utilise l'échantillon versionné ; remplacer les chemins pour travailler sur les fichiers Kaggle complets.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path('..')
tracks = pd.read_csv(ROOT / 'data/sample/tracks_sample.csv')
history = pd.read_csv(ROOT / 'data/sample/listening_history_sample.csv')
tracks.shape, history.shape

## Exploration, qualité et préparation pour les bases NoSQL

- nombreux genres manquants ;
- tags multivalués séparés par des virgules ;
- historique beaucoup plus volumineux que le catalogue ;
- forte concentration des écoutes ;
- nécessité de traiter l'historique par chunks ;
- besoin de dénormaliser les documents Firestore ;
- besoin de choisir les agrégats et classements à précharger dans Redis.

In [ ]:
missing = (tracks.isna().mean() * 100).sort_values(ascending=False)
missing.head(10)

In [ ]:
tracks['primary_genre'] = tracks['genre']
fallback = tracks['tags'].fillna('Unknown').str.split(',').str[0].str.strip()
tracks['primary_genre'] = tracks['primary_genre'].fillna(fallback)
tracks[['genre', 'tags', 'primary_genre']].head()

## Agrégations métier

In [ ]:
plays = history.groupby('track_id', as_index=False)['playcount'].sum()
top = plays.nlargest(20, 'playcount').merge(tracks[['track_id', 'name', 'artist']], on='track_id')
top[['name', 'artist', 'playcount']]

In [ ]:
top.sort_values('playcount').plot.barh(x='name', y='playcount', figsize=(10, 7), legend=False)
plt.title('Morceaux les plus écoutés dans l’échantillon')
plt.tight_layout();